# Train on the Figshare 141-point splits

Train one XASBlock per element from the Figshare train split. Early stopping selects each checkpoint using Figshare validation loss. The final table reports Figshare validation, Figshare test, and the original Materials Cloud test results. Each element gets separate 3×3 plots for Figshare and original test spectra; panels show spectra near the 10th through 90th percentile of per-spectrum test MSE.

The new split contains target spectra and IDs; this notebook computes the matching 64D features from its POSCAR files with the pinned local M3GNet model. Start Jupyter from this repository or `tutorial_omnixas`. Edit `DATA_ROOT` or `RUN_NAME` below if needed. The first run featurizes all new split structures and can take a while.


In [ ]:
from pathlib import Path
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pymatgen.core import Structure
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
if (cwd / 'tutorial_omnixas').is_dir():
    REPO_ROOT = cwd
elif cwd.name == 'tutorial_omnixas' and (cwd.parent / 'omnixas').is_dir():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError('Start Jupyter from the OmniXAS repository root or tutorial_omnixas directory.')
DATA_ROOT = REPO_ROOT.parent / 'OmniXAS_data'
NEW_ROOT = DATA_ROOT / 'figshare_141_splits'
OLD_ROOT = REPO_ROOT / 'tutorial_omnixas'
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
RUN_NAME = 'figshare_141_seed42'
RUN_DIR = REPO_ROOT / 'tutorial_omnixas' / 'output' / 'training' / RUN_NAME
SEED, MAX_EPOCHS, PATIENCE, BATCH_SIZE = 42, 300, 25, 128
HIDDEN_DIMS, LEARNING_RATE = [200, 200], 1e-3

required = [
    REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES' / 'model.pt',
    REPO_ROOT / 'omnixas' / 'featurizer' / 'm3gnet_featurizer.py',
]
for root in (NEW_ROOT / 'material_id_and_site', NEW_ROOT / 'ml_data', OLD_ROOT / 'material_id_and_site', OLD_ROOT / 'ml_data'):
    required.append(root)
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')
if RUN_DIR.exists():
    raise FileExistsError(f'Run directory already exists; change RUN_NAME to preserve prior results: {RUN_DIR}')

def read_ids(path):
    rows = [line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if not rows:
        raise ValueError(f'Empty ID split: {path}')
    return rows

def read_matrix(path, cols, label):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {label}: {path}')
    array = np.loadtxt(path, dtype=np.float32, ndmin=2)
    if array.shape[1] != cols or not np.isfinite(array).all():
        raise ValueError(f'{label} must be finite with {cols} columns; got {array.shape}: {path}')
    return array

# Validate all split files and exact ID/target/feature row alignment before loading M3GNet.
new_data, old_data = {}, {}
for element in ELEMENTS:
    for split in SPLITS:
        key = (element, split)
        new_ids = read_ids(NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        new_y = read_matrix(NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt', 141, 'new targets')
        if len(new_ids) != len(new_y):
            raise ValueError(f'New ID/target row mismatch for {key}: {len(new_ids)} != {len(new_y)}')
        old_ids = read_ids(OLD_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        old_X = read_matrix(OLD_ROOT / 'ml_data' / f'{element}_FEFF_{split}_X.txt', 64, 'old features')
        old_y = read_matrix(OLD_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt', 141, 'old targets')
        if len(old_ids) != len(old_X) or len(old_ids) != len(old_y):
            raise ValueError(f'Old ID/feature/target row mismatch for {key}: {len(old_ids)}, {len(old_X)}, {len(old_y)}')
        old_data[key] = {'ids': old_ids, 'X': old_X, 'y': old_y}
        new_data[key] = {'ids': new_ids, 'y': new_y}
        for value in new_ids:
            try:
                material_id, site = value.rsplit('_', 1)
                int(site)
            except ValueError as exc:
                raise ValueError(f'Invalid new split site ID {value!r} in {key}') from exc
            poscar = DATA_ROOT / 'figshare' / 'FEFF' / element / material_id / 'FEFF-XANES' / f'{int(site):03d}_{element}' / 'POSCAR'
            if not poscar.is_file():
                raise FileNotFoundError(f'No POSCAR for {value}: {poscar}')
        print(f'{element}: new train/val/test = {[len(new_data[(element, s)]["ids"]) for s in SPLITS]}, old = {[len(old_data[(element, s)]["ids"]) for s in SPLITS]}')

if RUN_DIR.exists():
    raise FileExistsError(f'Run directory was created while preparing inputs: {RUN_DIR}')


## Build the new split features

Each unique element/material structure is featurized once. Row order follows the split ID files exactly.


In [ ]:
from omnixas.featurizer.m3gnet_featurizer import M3GNetFeaturizer

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
featurizer = M3GNetFeaturizer(n_blocks=3)
structure_features = {}
for element in ELEMENTS:
    for split in SPLITS:
        for value in new_data[(element, split)]['ids']:
            material_id, site_text = value.rsplit('_', 1)
            cache_key = (element, material_id)
            if cache_key not in structure_features:
                poscar = DATA_ROOT / 'figshare' / 'FEFF' / element / material_id / 'FEFF-XANES' / f'{int(site_text):03d}_{element}' / 'POSCAR'
                structure = Structure.from_file(poscar)
                node_features = featurizer.featurize(structure)
                if node_features.ndim != 2 or node_features.shape[1] != 64:
                    raise ValueError(f'Expected 64D M3GNet node features for {poscar}; got {node_features.shape}')
                structure_features[cache_key] = node_features
new_X = {}
for key, data in new_data.items():
    element, _ = key
    rows = []
    for value in data['ids']:
        material_id, site_text = value.rsplit('_', 1)
        nodes = structure_features[(element, material_id)]
        site = int(site_text)
        if site >= len(nodes):
            raise IndexError(f'Site {site} is outside {material_id} structure with {len(nodes)} atoms')
        rows.append(nodes[site])
    X = np.asarray(rows, dtype=np.float32)
    if X.shape != (len(data['ids']), 64) or not np.isfinite(X).all():
        raise ValueError(f'Invalid new features for {key}: {X.shape}')
    data['X'] = X
    new_X[key] = X
print(f'Featurized {len(structure_features):,} unique element/material structures.')

## Train, validate, and evaluate

One head is trained for each element. The best validation-loss checkpoint is evaluated once on Figshare test and once on the original test split. Eta uses the mean spectrum from the new training set as its baseline for all three evaluations.


In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from omnixas.model.xasblock import XASBlock

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
XASBlock.DROPOUT = 0.5
RUN_DIR.mkdir(parents=True, exist_ok=False)
run_metadata = {
    'seed': SEED,
    'elements': ELEMENTS,
    'feature_source': 'M3GNet-MP-2021.2.8-PES, local pinned model; final 64D node features after 3 blocks',
    'encoder_path': str(REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES'),
    'target_source_and_scale': str(NEW_ROOT / 'report.json'),
    'target_grid': '141 points, 0.25 eV spacing; Figshare raw mu scaled using train-only Materials Cloud per-element median calibration',
    'head': {'hidden_dims': HIDDEN_DIMS, 'dropout': XASBlock.DROPOUT, 'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'early_stopping_patience': PATIENCE},
    'selection': 'minimum Figshare validation MSE; test evaluated after checkpoint selection',
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
records, predictions = [], {}

def predict(model, X):
    model.eval()
    output = []
    with torch.inference_mode():
        for start in range(0, len(X), BATCH_SIZE):
            xb = torch.as_tensor(X[start:start + BATCH_SIZE], dtype=torch.float32, device=device)
            output.append(model(xb).cpu().numpy())
    return np.concatenate(output, axis=0)

def score(element, split_name, y_true, y_pred, train_mean):
    per_spectrum_mse = np.mean((y_true - y_pred) ** 2, axis=1)
    baseline_per_spectrum = np.mean((y_true - train_mean) ** 2, axis=1)
    return {
        'element': element, 'evaluation_split': split_name,
        'n_spectra': len(y_true),
        'aggregate_mse': float(np.mean((y_true - y_pred) ** 2)),
        'median_spectrum_mse': float(np.median(per_spectrum_mse)),
        'baseline_median_mse': float(np.median(baseline_per_spectrum)),
        'eta': float(np.median(baseline_per_spectrum) / np.median(per_spectrum_mse)),
    }, per_spectrum_mse

for element in ELEMENTS:
    train, val = new_data[(element, 'train')], new_data[(element, 'val')]
    if len(train['X']) < 2 or len(val['X']) < 1:
        raise ValueError(f'Insufficient train/validation rows for {element}')
    model = XASBlock(input_dim=64, hidden_dims=HIDDEN_DIMS, output_dim=141).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TensorDataset(torch.from_numpy(train['X']), torch.from_numpy(train['y'])), batch_size=BATCH_SIZE, shuffle=True, drop_last=True, generator=generator)
    best_loss, best_state, stale = float('inf'), None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.mse_loss(model(xb), yb)
            loss.backward(); optimizer.step()
        model.eval(); val_total, val_rows = 0.0, 0
        with torch.inference_mode():
            for start in range(0, len(val['X']), BATCH_SIZE):
                xb = torch.as_tensor(val['X'][start:start + BATCH_SIZE], dtype=torch.float32, device=device)
                yb = torch.as_tensor(val['y'][start:start + BATCH_SIZE], dtype=torch.float32, device=device)
                val_total += nn.functional.mse_loss(model(xb), yb, reduction='sum').item(); val_rows += len(xb)
        val_loss = val_total / val_rows
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f'{element}: epoch {epoch:03d}, val MSE={val_loss:.6g}, best={best_loss:.6g}')
        if stale >= PATIENCE:
            break
    if best_state is None:
        raise RuntimeError(f'No validation checkpoint produced for {element}')
    model.load_state_dict(best_state)
    torch.save({'state_dict': best_state, 'element': element, 'seed': SEED, 'best_val_mse': best_loss, 'hidden_dims': HIDDEN_DIMS}, RUN_DIR / f'{element}_best.pt')
    train_mean = train['y'].mean(axis=0, keepdims=True)
    for split_name, data in [('new_val', val), ('new_test', new_data[(element, 'test')]), ('old_test', old_data[(element, 'test')])]:
        y_pred = predict(model, data['X'])
        row, errors = score(element, split_name, data['y'], y_pred, train_mean)
        records.append(row)
        predictions[(element, split_name)] = {'y': data['y'], 'pred': y_pred, 'errors': errors}
    print(f'{element}: restored best validation checkpoint at MSE {best_loss:.6g}')

results = pd.DataFrame(records).sort_values(['element', 'evaluation_split'])
results.to_csv(RUN_DIR / 'evaluation.csv', index=False)
display(results.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))

## Test spectrum percentiles

Each panel shows the target and prediction for the spectrum nearest that error percentile, with spectra ranked by per-spectrum MSE. Separate figures keep the new and original test sets easy to compare.


In [ ]:
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90]
energy = np.arange(141) * 0.25
for element in ELEMENTS:
    for split_name in ('new_test', 'old_test'):
        item = predictions[(element, split_name)]
        order = np.argsort(item['errors'])
        fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey=True)
        for ax, percentile in zip(axes.flat, percentiles):
            idx = order[int(round((percentile / 100) * (len(order) - 1)))]
            ax.plot(energy, item['y'][idx], label='target', linewidth=1.5)
            ax.plot(energy, item['pred'][idx], label='prediction', linewidth=1.2)
            ax.set_title(f'{percentile}th percentile (MSE={item["errors"][idx]:.3g})')
            ax.grid(alpha=0.2)
        fig.suptitle(f'{element} — {split_name.replace("_", " ")}')
        fig.supxlabel('Energy above grid start (eV)')
        fig.supylabel('Intensity')
        axes.flat[0].legend(frameon=False)
        fig.tight_layout()
        fig.savefig(RUN_DIR / f'{element}_{split_name}_percentiles.png', dpi=140, bbox_inches='tight')
        plt.show()
        plt.close(fig)
print(f'Results and plots saved in {RUN_DIR}')